In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Objective:
- Verify tokenization of dialogue–summary pairs
- Inspect input_ids, attention_mask, and labels
- Validate truncation and padding behavior
- Catch issues before model training

In [2]:
!pip install -r /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 8.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=57231b466693d295d8cbc4817371b33521263e2963b6d8031500736ae1172b95
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [3]:
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer


In [4]:
RAW_DATA_DIR = Path("/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw")

data_files = {
    "train": str(RAW_DATA_DIR / "samsum_train.json"),
}
dataset = load_dataset("json", data_files=data_files)
dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
})

In [5]:
sample = dataset["train"][0]

print("Dialogue:\n")
print(sample["dialogue"])

print("\nSummary:\n")
print(sample["summary"])

Dialogue:

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:

Amanda baked cookies and will bring Jerry some tomorrow.


In [6]:
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-large")

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [7]:
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128

In [8]:
inputs = tokenizer(
    sample["dialogue"],
    max_length=MAX_INPUT_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt",
)
inputs

{'input_ids': tensor([[12195,   151,   125,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0]])}

In [9]:
decoded_dialogue = tokenizer.decode(
    inputs["input_ids"][0],
    skip_special_tokens=True,
)
decoded_dialogue[:500]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

In [14]:
labels = tokenizer(
    text_target=sample["summary"],
    max_length=MAX_TARGET_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt",
)
labels

{'input_ids': tensor([[12195,  7091,  3659,   111,   138,   650, 10508,   181,  3469,   107,
             1,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [11]:
decoded_summary = tokenizer.decode(
    labels["input_ids"][0],
    skip_special_tokens=True,
)
decoded_summary

'Amanda baked cookies and will bring Jerry some tomorrow.'

In [12]:
inputs["attention_mask"][0][-20:]

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [13]:
training_sample = {
    "input_ids": inputs["input_ids"][0],
    "attention_mask": inputs["attention_mask"][0],
    "labels": labels["input_ids"][0],
}
training_sample

{'input_ids': tensor([12195,   151,   125,  ...,     0,     0,     0]),
 'attention_mask': tensor([1, 1, 1,  ..., 0, 0, 0]),
 'labels': tensor([12195,  7091,  3659,   111,   138,   650, 10508,   181,  3469,   107,
             1,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,    